# Malware Removal Package Builder (No-Triage Mode)

This notebook hardens and translates a no-triage malware removal procedure into a deployable package for Windows execution from a healthy WSL2 context.

## What this notebook produces
- `removal_package/iocs.json`: known-bad indicators
- `removal_package/remediate_windows.ps1`: hardened removal script
- `removal_package/manifest.json`: SHA-256 integrity manifest

## Safety defaults
- Script supports `Audit` mode and defaults to non-destructive dry behavior unless explicitly run with `-Mode Execute`
- Administrator rights are enforced in the PowerShell script
- Script hash can be verified at runtime with `-ExpectedHash`

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
from pathlib import Path

BUILD_TIME = dt.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
ROOT = Path.cwd()
PACKAGE_DIR = ROOT / 'removal_package'
PACKAGE_DIR.mkdir(exist_ok=True)

IOC_FILE = PACKAGE_DIR / 'iocs.json'
PS1_FILE = PACKAGE_DIR / 'remediate_windows.ps1'
MANIFEST_FILE = PACKAGE_DIR / 'manifest.json'

print('Package directory:', PACKAGE_DIR)
print('Build time:', BUILD_TIME)

## 1) Configure known indicators

Populate these values with known malicious objects observed in your environment. Since triage is intentionally skipped, these lists are the deterministic basis for removal.

In [ ]:
ioc_payload = {
    'schema_version': 1,
    'generated_utc': BUILD_TIME,
    'processes': [
        # 'badproc.exe',
    ],
    'services': [
        # 'BadService',
    ],
    'scheduled_tasks': [
        # '\\Microsoft\\Windows\\Update\\BadTask',
    ],
    'paths': [
        # r'C:\\ProgramData\\BadFolder',
        # r'C:\\Users\\Public\\bad.exe',
    ],
    'run_values': [
        # {'key': r'HKCU:\\Software\\Microsoft\\Windows\\CurrentVersion\\Run', 'name': 'BadValue'},
        # {'key': r'HKLM:\\Software\\Microsoft\\Windows\\CurrentVersion\\Run', 'name': 'BadValue'},
    ]
}

IOC_FILE.write_text(json.dumps(ioc_payload, indent=2), encoding='utf-8')
print('Wrote', IOC_FILE)
print(json.dumps({k: len(v) if isinstance(v, list) else v for k, v in ioc_payload.items() if k != 'generated_utc'}, indent=2))

## 2) Build hardened PowerShell removal script

Hardening details in this generated script:
- strict mode and stop-on-error behavior
- mandatory admin check
- transcript logging
- runtime script hash validation option
- audit mode for non-destructive preview
- deterministic kill/remove sequence

In [ ]:
ps_script = r'''
param(
    [Parameter(Mandatory = $true)]
    [string]$IocFile,

    [ValidateSet('Audit','Execute')]
    [string]$Mode = 'Audit',

    [string]$ExpectedHash = ''
)

Set-StrictMode -Version Latest
$ErrorActionPreference = 'Stop'
$ProgressPreference = 'SilentlyContinue'

function Assert-Admin {
    $identity = [Security.Principal.WindowsIdentity]::GetCurrent()
    $principal = New-Object Security.Principal.WindowsPrincipal($identity)
    if (-not $principal.IsInRole([Security.Principal.WindowsBuiltInRole]::Administrator)) {
        throw 'Administrator privileges are required for this removal script.'
    }
}

function Assert-ScriptIntegrity {
    param([string]$Expected)
    if ([string]::IsNullOrWhiteSpace($Expected)) { return }

    $actual = (Get-FileHash -Path $PSCommandPath -Algorithm SHA256).Hash.ToLowerInvariant()
    $exp = $Expected.ToLowerInvariant()
    if ($actual -ne $exp) {
        throw "Script hash mismatch. Expected $exp but got $actual"
    }
}

function Read-Iocs {
    param([string]$Path)
    if (-not (Test-Path -LiteralPath $Path)) {
        throw "IOC file not found: $Path"
    }
    return Get-Content -LiteralPath $Path -Raw | ConvertFrom-Json
}

function Stop-BadProcesses {
    param($Names, [bool]$Apply)
    foreach ($n in $Names) {
        $base = [System.IO.Path]::GetFileNameWithoutExtension([string]$n)
        if ([string]::IsNullOrWhiteSpace($base)) { continue }
        if ($Apply) {
            Get-Process -Name $base -ErrorAction SilentlyContinue | Stop-Process -Force -ErrorAction SilentlyContinue
        } else {
            Write-Host "[AUDIT] Would stop process name: $base"
        }
    }
}

function Disable-BadServices {
    param($Names, [bool]$Apply)
    foreach ($svc in $Names) {
        if ($Apply) {
            Stop-Service -Name $svc -Force -ErrorAction SilentlyContinue
            Set-Service -Name $svc -StartupType Disabled -ErrorAction SilentlyContinue
        } else {
            Write-Host "[AUDIT] Would stop/disable service: $svc"
        }
    }
}

function Remove-BadTasks {
    param($TaskPaths, [bool]$Apply)
    foreach ($full in $TaskPaths) {
        $f = [string]$full
        if (-not $f.Contains('\')) { continue }
        $idx = $f.LastIndexOf('\')
        $taskPath = $f.Substring(0, $idx + 1)
        $taskName = $f.Substring($idx + 1)
        if ($Apply) {
            Disable-ScheduledTask -TaskPath $taskPath -TaskName $taskName -ErrorAction SilentlyContinue
            Unregister-ScheduledTask -TaskPath $taskPath -TaskName $taskName -Confirm:$false -ErrorAction SilentlyContinue
        } else {
            Write-Host "[AUDIT] Would disable/unregister task: $f"
        }
    }
}

function Remove-BadRunValues {
    param($Entries, [bool]$Apply)
    foreach ($entry in $Entries) {
        $key = [string]$entry.key
        $name = [string]$entry.name
        if ([string]::IsNullOrWhiteSpace($key) -or [string]::IsNullOrWhiteSpace($name)) { continue }
        if ($Apply) {
            Remove-ItemProperty -Path $key -Name $name -Force -ErrorAction SilentlyContinue
        } else {
            Write-Host "[AUDIT] Would remove Run value: $key :: $name"
        }
    }
}

function Remove-BadPaths {
    param($Paths, [bool]$Apply)
    foreach ($p in $Paths) {
        $target = [string]$p
        if ([string]::IsNullOrWhiteSpace($target)) { continue }
        if (-not (Test-Path -LiteralPath $target)) { continue }
        if ($Apply) {
            takeown /f $target /r /d y | Out-Null
            icacls $target /grant Administrators:F /t /c | Out-Null
            Remove-Item -LiteralPath $target -Recurse -Force -ErrorAction SilentlyContinue
        } else {
            Write-Host "[AUDIT] Would remove path: $target"
        }
    }
}

function Invoke-DefenderCleanup {
    param([bool]$Apply)
    if ($Apply) {
        Update-MpSignature
        Start-MpScan -ScanType QuickScan
        Start-MpScan -ScanType FullScan
    } else {
        Write-Host '[AUDIT] Would run Update-MpSignature, QuickScan, FullScan'
    }
}

Assert-Admin
Assert-ScriptIntegrity -Expected $ExpectedHash

$logDir = Join-Path $env:ProgramData 'MalwareRemoval'
New-Item -ItemType Directory -Path $logDir -Force | Out-Null
$stamp = Get-Date -Format 'yyyyMMdd_HHmmss'
$transcript = Join-Path $logDir "removal_$stamp.log"
Start-Transcript -Path $transcript -Force

try {
    $iocs = Read-Iocs -Path $IocFile
    $apply = $Mode -eq 'Execute'

    Write-Host "Mode: $Mode"
    Stop-BadProcesses -Names $iocs.processes -Apply $apply
    Disable-BadServices -Names $iocs.services -Apply $apply
    Remove-BadTasks -TaskPaths $iocs.scheduled_tasks -Apply $apply
    Remove-BadRunValues -Entries $iocs.run_values -Apply $apply
    Remove-BadPaths -Paths $iocs.paths -Apply $apply
    Invoke-DefenderCleanup -Apply $apply

    Write-Host 'Completed no-triage removal sequence.'
    Write-Host 'Reboot and validate recurrence status after execution.'
}
finally {
    Stop-Transcript
}
'''

PS1_FILE.write_text(ps_script, encoding='utf-8', newline='
')
print('Wrote', PS1_FILE)

## 3) Build integrity manifest

This manifest is consumed by the deployer to verify package integrity before deployment and execution.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    'package_version': 1,
    'generated_utc': BUILD_TIME,
    'files': [
        'iocs.json',
        'remediate_windows.ps1'
    ],
    'hashes': {
        'iocs.json': sha256(IOC_FILE),
        'remediate_windows.ps1': sha256(PS1_FILE)
    }
}

MANIFEST_FILE.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Wrote', MANIFEST_FILE)
print(json.dumps(manifest, indent=2))

## 4) Deployment commands

Use the Python deployer script from WSL:

- Deploy only (safe default):
  - `python3 deploy_malware_removal.py`
- Deploy and run in Audit mode with elevation:
  - `python3 deploy_malware_removal.py --execute --mode Audit --elevate`
- Deploy and run live removal (explicit guard required):
  - `python3 deploy_malware_removal.py --execute --mode Execute --allow-live --elevate`

After live execution, reboot and verify no process/service/task/path recurrence.

## 5) Schedule weekly maintenance scans

Register a Windows Task Scheduler task and a WSL cron entry so that:
- Windows Defender runs a full signature-updated scan every Sunday at the configured hour
- The WSL cross-boundary intelligence scan (`wsl_scan.py`) runs in the same window
- WSL stays **online and accessible** during scan hours — it acts as an independent sensor
- All findings are logged to `/mnt/c/ProgramData/MalwareRemoval/` and are readable from Windows
- If `wsl_scan.py` scores above the severity threshold a `RESET_RECOMMENDED.flag` file is created for operator review

Command (from WSL, no IOC package required):
```
python3 deploy_malware_removal.py --schedule
```
Custom off-peak hour (e.g. 02:00):
```
python3 deploy_malware_removal.py --schedule --scan-hour 2
```